<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m3_02_automated_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M3.02 — Letting scikit-learn search

In the lab we replaced our own loops with `GridSearchCV` and
`RandomizedSearchCV`, and we saw that the score of a tuned model must be
measured with a second, outer cross-validation. In this workbook you run
these searches **on the spam filter**. You will look at what two
hyperparameters do to a model, compare the cost of three search
strategies, and catch a search that finds a "good" model in pure noise.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "C_when_gamma_smaller": "4f627c9e32f4ff3e",
    "honest_score": "233562de1a0288b1",
    "n_candidates": "8527a891e2241369",
    "n_fits_random": "16dc368a89b428b2",
    "n_second_round": "4a44dc15364204a8",
    "n_total_fits": "7f2253d7e228b22a",
    "top_two_distinguishable": "9390298f3fb0c5b1"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and pipeline

The same e-mails as in workbook M3.01: 57 counts per e-mail, and
`is_spam` to predict. We keep the pipeline that won there: logarithm,
scaling, logistic regression.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

spam = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/spambase.csv"
)
data = spam.drop(columns="is_spam")
target = spam["is_spam"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

model = make_pipeline(
    FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=1000)
)

## 2. Grid search

`GridSearchCV` does what our double loop did in workbook M3.01: it tries
**every combination** of the values we list, cross-validates each one,
and then trains the best combination once more on all the data
(`refit=True`, the default), so that the search object can be used as a
model.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "functiontransformer": [FunctionTransformer(np.log1p), "passthrough"],
    "logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
}
grid_search = GridSearchCV(model, param_grid=param_grid, cv=cv)

✏️ **Before running it:** how many combinations (candidates) will the
search try, and how many times will it train the pipeline in total? Do
not forget the final training on all the data.

In [ ]:
n_candidates = ____
n_total_fits = ____
check("n_candidates", n_candidates)
check("n_total_fits", n_total_fits, hint="Candidates × folds, plus the final refit.")

In [ ]:
grid_search.fit(data, target)
results = pd.DataFrame(grid_search.cv_results_)
results["log"] = results["param_functiontransformer"].astype(str).str.contains("log1p")
top = results.sort_values("rank_test_score")[
    ["log", "param_logisticregression__C", "mean_test_score", "std_test_score", "rank_test_score"]
].head(5)
top.round(4)

`cv_results_` holds one row per candidate. `mean_test_score` is the mean
accuracy over the 5 folds and `std_test_score` how much it varied from
fold to fold.

✏️ Compare the difference between the candidates ranked 1 and 2 with
their `std_test_score`. Does this search tell us reliably which of the
two is better? Write `"yes"` or `"no"`.

In [ ]:
top_two_distinguishable = ____
check("top_two_distinguishable", top_two_distinguishable)

`best_params_` reports the candidate ranked 1 as if it were *the*
answer. It is the winner of a close race between candidates
that are, as far as we can measure, equally good. scikit-learn's example
[Statistical comparison of models using grid search](https://scikit-learn.org/1.8/auto_examples/model_selection/plot_grid_search_stats.html)
shows how to test such differences properly.

🧠 **Think.** Among candidates you cannot tell apart, which one would you
put in production? What else than accuracy could decide?

## 3. ➕ Going further: seeing what two hyperparameters do

A logistic regression draws a straight border between spam and normal
e-mails. A **support vector machine** with an RBF kernel (`SVC`) can draw
curved borders. Two hyperparameters shape them:

- `gamma`: how far the influence of one e-mail reaches. Small: far, the
  border is smooth. Large: only the immediate surroundings.
- `C`: how hard the model tries to classify every training e-mail
  correctly. Small: it tolerates errors and stays simple.

To **see** the borders we use only 300 e-mails and two features: the
total number of capital letters and the average length of a sequence of
capitals.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.svm import SVC

sample = spam.sample(n=300, random_state=0)
two_features = ["capital_run_length_total", "capital_run_length_average"]
data_2d = np.log1p(sample[two_features])
target_2d = sample["is_spam"]

🔮 **Predict first.** We draw the border for 3 values of `gamma`. For
which one will the model draw small **islands** around single e-mails?
`"smallest gamma"`, `"middle gamma"` or `"largest gamma"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["smallest gamma", "middle gamma", "largest gamma"])

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 10), sharex=True, sharey=True)
for row, gamma in zip(axes, [0.05, 2, 50]):
    for ax, C in zip(row, [0.1, 10, 1000]):
        svm = make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma)).fit(data_2d, target_2d)
        DecisionBoundaryDisplay.from_estimator(
            svm, data_2d, response_method="predict", alpha=0.4, cmap="coolwarm", ax=ax
        )
        ax.scatter(
            data_2d.iloc[:, 0], data_2d.iloc[:, 1], c=target_2d, cmap="coolwarm", s=10, edgecolor="black", linewidth=0.2
        )
        ax.set(title=f"gamma={gamma}, C={C}: training accuracy {svm.score(data_2d, target_2d):.2f}", xlabel="", ylabel="")
fig.supxlabel("log(1 + total number of capital letters)")
_ = fig.supylabel("log(1 + average length of a run of capitals)")

Red is "spam", blue is "normal". From top to bottom (`gamma` grows) and
from left to right (`C` grows), the border goes from almost straight to
a patchwork that isolates single e-mails: from **underfitting** to
**overfitting**. The training accuracy in each title climbs towards 1 in
the bottom-right corner, and you know by now what that is worth.

One panel does not fit this reading: bottom left, with the largest
`gamma` and the smallest `C`, the model is allowed so little that it
predicts the same class almost everywhere. The two settings have to be
read together.

Now the full problem: all 57 features, 1000 e-mails, and a grid of 6
values for each hyperparameter. The colour is the mean cross-validated
accuracy.

In [ ]:
subset = spam.sample(n=1000, random_state=0)
data_subset, target_subset = subset.drop(columns="is_spam"), subset["is_spam"]
svm = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), SVC())
svm_grid = {"svc__C": np.logspace(-2, 3, 6), "svc__gamma": np.logspace(-5, 0, 6)}
svm_search = GridSearchCV(svm, param_grid=svm_grid, cv=cv).fit(data_subset, target_subset)

heatmap = pd.DataFrame(svm_search.cv_results_).pivot(
    index="param_svc__gamma", columns="param_svc__C", values="mean_test_score"
)
fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(heatmap, cmap="viridis", origin="lower")
ax.set_xticks(range(6), [f"{C:g}" for C in heatmap.columns])
ax.set_yticks(range(6), [f"{gamma:g}" for gamma in heatmap.index])
ax.set(xlabel="C", ylabel="gamma", title="Cross-validated accuracy of the SVM")
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{heatmap.iloc[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)
_ = fig.colorbar(image)

The good models (yellow) form a **diagonal band**: the two
hyperparameters compensate each other.

✏️ Follow the band. When `gamma` gets **smaller**, does the best `C` get
`"smaller"` or `"larger"`?

In [ ]:
C_when_gamma_smaller = ____
check("C_when_gamma_smaller", C_when_gamma_smaller)

This is why hyperparameters must be searched **together**: the best `C`
depends on `gamma`. Tuning one, fixing it and then tuning the other can
end far from the band. The picture follows scikit-learn's example
[RBF SVM parameters](https://scikit-learn.org/1.8/auto_examples/svm/plot_rbf_parameters.html).

## 4. Randomized search

The grid above needed 36 candidates, most of them in the dark corners. A
randomized search draws candidates **at random** from ranges we give, and
we decide how many. `loguniform(a, b)` draws values between `a` and `b`
so that every order of magnitude is equally likely, which is the random
version of a logarithmic grid.

✏️ We ask for 20 candidates, on 5 folds. How many times is the pipeline
trained in total (again with the final refit)?

In [ ]:
n_fits_random = ____
check("n_fits_random", n_fits_random)

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

distributions = {"svc__C": loguniform(1e-2, 1e3), "svc__gamma": loguniform(1e-5, 1e0)}
random_search = RandomizedSearchCV(
    svm, param_distributions=distributions, n_iter=20, cv=cv, random_state=0
).fit(data_subset, target_subset)
print(f"grid search,       36 candidates: best accuracy {svm_search.best_score_:.3f}")
print(f"randomized search, 20 candidates: best accuracy {random_search.best_score_:.3f}")

With little more than half the work, the randomized search finds a model
about as good. The advantage grows with the number of hyperparameters: a grid
with 6 values for each of 5 hyperparameters has 7776 candidates, while a
randomized search still tries exactly as many as you ask for.

### ➕ Going further: stop wasting time on bad candidates

Most candidates are obviously bad after a quick look. **Successive
halving** organises a tournament: all candidates are first evaluated on
a **small part** of the data, the better half goes on to a round with
twice as much data, and so on until few are left.

✏️ We start with 20 candidates and keep the better half after each
round (`factor=2`). How many candidates take part in the **second**
round?

In [ ]:
n_second_round = ____
check("n_second_round", n_second_round)

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

halving_search = HalvingRandomSearchCV(
    svm, param_distributions=distributions, n_candidates=20, factor=2, cv=cv, random_state=0
).fit(data_subset, target_subset)
pd.DataFrame(
    {"candidates": halving_search.n_candidates_, "e-mails used": halving_search.n_resources_},
    index=pd.RangeIndex(1, halving_search.n_iterations_ + 1, name="round"),
)

The many weak candidates are eliminated cheaply, on a few dozen e-mails;
only the finalists are trained on a large part of the data. The import
line with `enable_halving_search_cv` is needed because scikit-learn
still considers this tool experimental.

## 5. ➕ Going further: the best of many is too good to be true

Here is an experiment with a known answer. We take 200 e-mails, half of
them spam, and we **shuffle the labels**: every e-mail gets a label that
has nothing to do with its content. No model can learn anything from
this data. The honest accuracy of any model is 0.5, a coin toss.

In [ ]:
half = 100
noise_sample = pd.concat(
    [spam[spam["is_spam"] == 1].sample(half, random_state=0), spam[spam["is_spam"] == 0].sample(half, random_state=0)]
)
noise_data = noise_sample.drop(columns="is_spam")
noise_target = np.random.default_rng(0).permutation(noise_sample["is_spam"].to_numpy())

We "tune" a small decision tree on this noise. The 200 candidates differ
only by their random seed, which changes the features each tree happens
to look at. None of them can be better than another.

🔮 **Predict first.** What will the search report as the accuracy of its
best candidate (`best_score_`)? **"below 0.45"**, **"about 0.5"** or
**"above 0.55"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["below 0.45", "about 0.5", "above 0.55"])

In [ ]:
from sklearn.tree import DecisionTreeClassifier

noise_search = GridSearchCV(
    DecisionTreeClassifier(max_depth=3, max_features=5),
    param_grid={"random_state": list(range(200))},
    cv=5,
)
noise_search.fit(noise_data, noise_target)
print(f"accuracy of the best candidate according to the search: {noise_search.best_score_:.3f}")

The search has "found" a model that seems to beat a coin toss, on data
where that is impossible. Among 200 candidates that are all at 0.5 on
average, some are lucky on these particular folds, and the search picks
the luckiest.

The remedy from the lab is **nested cross-validation**: the whole search
is treated as the model, and evaluated on e-mails that took no part in
it.

In [ ]:
from sklearn.model_selection import cross_val_score

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
nested_scores = cross_val_score(noise_search, noise_data, noise_target, cv=outer_cv)
print(f"best_score_ of the search: {noise_search.best_score_:.3f}")
print(f"nested cross-validation:   {nested_scores.mean():.3f} ± {nested_scores.std():.3f}")

✏️ Which of the two numbers tells the truth about how the tuned model
will do on new e-mails? Write `"best_score_"` or `"nested"`.

In [ ]:
honest_score = ____
check("honest_score", honest_score)

Now the real spam filter of section 2, with real labels:

In [ ]:
real_nested = cross_val_score(grid_search, data, target, cv=outer_cv)
print(f"best_score_ of the search: {grid_search.best_score_:.3f}")
print(f"nested cross-validation:   {real_nested.mean():.3f} ± {real_nested.std():.3f}")

🧠 **Think.**

1. On the real data the two numbers are close; on the noise they were
   far apart. Name two differences between the two searches that explain
   it. (Hint: how many candidates, how many e-mails, and is there
   something to learn?)
2. A colleague tunes the random seed of a model "because seed 137 gives
   2% more accuracy". What do you tell them?
3. After a search, you evaluate the best model on a test set, find the
   score disappointing, change the grid, search again, and repeat until
   the test score is good. What has the test set become?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - what a grid search does, what it costs, and why its first-ranked
     candidate is not "the" answer;
   - the nine borders of the SVM and the diagonal band of the heat map;
   - grid search, randomized search and successive halving: what each
     one spends its trainings on;
   - the search on shuffled labels: why it reports more than 0.5 and how
     nested cross-validation corrects it.